In [1]:
"""
generate_sample_data.py
-----------------------
Generates a realistic mock product dataset for the attribution engine.
This script creates a CSV file with product details, including some intentional data quality issues 
to simulate real-world scenarios. It also saves the client taxonomy in a JSON file for use in the attribution process.
"""

import pandas as pd
import numpy as np
import json
import os

np.random.seed(42)

# ── Sample product data ──────────────────────────────────────────────────────

PRODUCTS = [
    # Skincare
    ("Nivea Q10 Anti-Wrinkle Day Cream 50ml", "Nivea", "personal care"),
    ("Olay Regenerist Micro-Sculpting Cream", "Olay", "skincare"),
    ("Neutrogena Hydro Boost Water Gel", "Neutrogena", "moisturiser"),
    ("Cetaphil Gentle Skin Cleanser 500ml", "Cetaphil", "face wash"),
    ("Lakme Absolute Skin Natural Mousse", "Lakme", None),
    ("Pond's White Beauty Cream 35g", "Pond's", "skincare"),
    ("Garnier Light Complete Serum Cream", "Garnier", "cream"),
    ("The Body Shop Vitamin E Moisturiser", "The Body Shop", "skin"),
    ("Himalaya Herbals Nourishing Skin Cream", "Himalaya", None),
    ("Biotique Bio Almond Oil Moisturiser", "Biotique", "moisturiser"),

    # Haircare
    ("Head & Shoulders Anti-Dandruff Shampoo 400ml", "P&G", "hair/shampoo"),
    ("Pantene Pro-V Silky Smooth Shampoo", "Pantene", "shampoo"),
    ("Dove Intense Repair Shampoo 340ml", "Dove", "hair care"),
    ("TRESemmé Keratin Smooth Shampoo", "TRESemmé", "hair"),
    ("Sunsilk Stunning Black Shine Shampoo", "Sunsilk", None),
    ("Garnier Ultra Blends Mythic Olive Conditioner", "Garnier", "conditioner"),
    ("Dove Intense Repair Conditioner", "Dove", "hair"),
    ("Parachute Advansed Jasmine Hair Oil 300ml", "Parachute", "hair oil"),
    ("Indulekha Bringha Hair Oil", "Indulekha", None),
    ("Himalaya Anti-Dandruff Hair Cream", "Himalaya", "hair"),

    # Bath & Body
    ("Dove Deeply Nourishing Body Wash 500ml", "Dove", "personal care"),
    ("Lux Velvet Touch Body Wash", "Lux", "body wash"),
    ("Dettol Original Liquid Hand Wash 200ml", "Dettol", "soap/hygiene"),
    ("Lifebuoy Total 10 Body Wash", "Lifebuoy", "hygiene"),
    ("Pears Pure & Gentle Shower Gel", "Pears", "bath"),
    ("Palmolive Naturals Milk & Honey Body Wash", "Palmolive", None),
    ("Fiama Di Wills Peach & Avocado Shower Gel", "Fiama", "body"),
    ("Savlon Moisturising Hand Wash 200ml", "Savlon", "hygiene"),
    ("Biotique Morning Nectar Body Lotion", "Biotique", "lotion"),
    ("Vaseline Intensive Care Body Lotion", "Vaseline", "body lotion"),

    # Oral Care
    ("Colgate MaxFresh Toothpaste 150g", "Colgate", "oral care"),
    ("Sensodyne Rapid Relief Toothpaste", "Sensodyne", "toothpaste"),
    ("Oral-B Pro-Expert Toothbrush", "Oral-B", "dental"),
    ("Colgate 360 Charcoal Gold Toothbrush", "Colgate", "brush"),
    ("Listerine Cool Mint Mouthwash 500ml", "Listerine", None),
    ("Dabur Red Toothpaste Ayurvedic", "Dabur", "oral"),
    ("Pepsodent Germicheck Toothpaste", "Pepsodent", None),
    ("Close Up Deep Action Toothpaste", "Close Up", "toothpaste"),

    # Ambiguous / edge cases
    ("Himalaya Wellness Complete Care", "Himalaya", None),       # brand only
    ("Generic Product 001", "Unknown", None),                   # fully unknown
    ("Dove Men+Care Face Wash", "Dove", "men care"),             # cross-category
    ("Johnson's Baby Powder 200g", "Johnson's", None),           # unclear adult/baby
    ("Vicks VapoRub 50ml", "Vicks", "wellness"),                 # out-of-scope
    ("Gillette Mach3 Razor", "Gillette", "grooming"),            # out of taxonomy
    ("Old Spice After Shave Lotion", "Old Spice", None),         # ambiguous
]

def generate_data():
    os.makedirs("data", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    records = []
    for i, (name, brand, raw_cat) in enumerate(PRODUCTS):
        records.append({
            "product_id": f"P{str(i+1).zfill(4)}",
            "product_name": name,
            "brand": brand,
            "raw_category": raw_cat,
            "price_inr": round(np.random.uniform(50, 800), 2),
            "stock_units": np.random.randint(0, 500),
        })

    df = pd.DataFrame(records)

    # Introduce realistic data quality issues
    df.loc[df.sample(5, random_state=1).index, "raw_category"] = None
    df.loc[df.sample(3, random_state=2).index, "brand"] = None
    df.loc[df.sample(2, random_state=3).index, "product_name"] = df.loc[
        df.sample(2, random_state=3).index, "product_name"
    ]  # keep as-is (simulate duplicate names)

    # Add one actual duplicate row
    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

    df.to_csv("data/raw_products.csv", index=False)
    print(f"Generated {len(df)} product records → data/raw_products.csv")
    print(f"  Null raw_category: {df['raw_category'].isnull().sum()}")
    print(f"  Null brand: {df['brand'].isnull().sum()}")
    print(f"  Duplicates: {df.duplicated().sum()}")


# ── Client taxonomy definition ───────────────────────────────────────────────

TAXONOMY = {
    "Skincare": {
        "keywords": [
            "moisturiser", "moisturizer", "cream", "serum", "cleanser",
            "face wash", "sunscreen", "spf", "toner", "skin", "lotion face",
            "anti-wrinkle", "whitening", "brightening", "gel face"
        ],
        "brands": ["Nivea", "Olay", "Neutrogena", "Cetaphil", "Lakme",
                   "Pond's", "Garnier", "The Body Shop", "Himalaya", "Biotique"]
    },
    "Haircare": {
        "keywords": [
            "shampoo", "conditioner", "hair oil", "hair cream", "hair mask",
            "hair serum", "dandruff", "keratin", "hair", "scalp"
        ],
        "brands": ["Head & Shoulders", "Pantene", "TRESemmé", "Sunsilk",
                   "Parachute", "Indulekha"]
    },
    "Bath & Body": {
        "keywords": [
            "body wash", "shower gel", "hand wash", "soap", "bath",
            "body lotion", "body milk", "hygiene", "liquid wash", "scrub body"
        ],
        "brands": ["Dove", "Lux", "Dettol", "Lifebuoy", "Pears",
                   "Palmolive", "Fiama", "Savlon", "Vaseline"]
    },
    "Oral Care": {
        "keywords": [
            "toothpaste", "toothbrush", "mouthwash", "dental", "teeth",
            "whitening teeth", "oral", "floss", "gum", "brush teeth"
        ],
        "brands": ["Colgate", "Sensodyne", "Oral-B", "Listerine",
                   "Dabur", "Pepsodent", "Close Up"]
    }
}

def save_taxonomy():
    with open("data/client_taxonomy.json", "w") as f:
        json.dump(TAXONOMY, f, indent=2)
    print("Saved taxonomy → data/client_taxonomy.json")


if __name__ == "__main__":
    generate_data()
    save_taxonomy()

Generated 46 product records → data/raw_products.csv
  Null raw_category: 16
  Null brand: 3
  Duplicates: 1
Saved taxonomy → data/client_taxonomy.json


In [2]:
"""
product_attribution.py
----------------------
Product Attribution & Category Mapping Engine

Maps raw product data to a client-defined category taxonomy using:
  1. Keyword scoring (HIGH confidence)
  2. Brand-level inference (MEDIUM confidence)
  3. Fallback flagging for manual review (LOW confidence)


"""

import pandas as pd
import numpy as np
import json
import re
import os
from datetime import datetime


# ── Configuration ────────────────────────────────────────────────────────────

INPUT_PRODUCTS  = "data/raw_products.csv"
INPUT_TAXONOMY  = "data/client_taxonomy.json"
OUTPUT_CSV      = "output/attributed_products.csv"
OUTPUT_REPORT   = "output/attribution_report.txt"

# Scoring thresholds — adjust based on taxonomy density
HIGH_THRESHOLD   = 1   # ≥1 keyword match → HIGH
MEDIUM_THRESHOLD = 1   # ≥1 brand match → MEDIUM
# Below both → LOW / NEEDS_REVIEW


# ── Step 1: Load & Audit ─────────────────────────────────────────────────────

def load_and_audit(filepath: str) -> pd.DataFrame:
    """
    Load raw product CSV and print a data quality audit.
    ASSUMPTION: product_id is the unique identifier.
    """
    print("\n" + "="*60)
    print("STEP 1: LOADING & AUDITING DATA")
    print("="*60)

    df = pd.read_csv(filepath)

    print(f"\nShape          : {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Columns        : {list(df.columns)}")
    print(f"\nNull counts:")
    print(df.isnull().sum().to_string())
    print(f"\nDuplicate rows : {df.duplicated().sum()}")
    print(f"Duplicate IDs  : {df.duplicated(subset='product_id').sum()}")

    # ASSUMPTION: drop full duplicate rows (exact same product twice)
    before = len(df)
    df = df.drop_duplicates()
    dropped = before - len(df)
    if dropped:
        print(f"\n⚠  Dropped {dropped} exact duplicate row(s)")

    # Normalise text columns
    df["product_name_clean"] = (
        df["product_name"]
        .fillna("")
        .str.lower()
        .str.strip()
        .apply(lambda x: re.sub(r"[^a-z0-9\s&+'-]", " ", x))
        .apply(lambda x: re.sub(r"\s+", " ", x).strip())
    )
    df["brand_clean"] = df["brand"].fillna("unknown").str.strip().str.lower()
    df["raw_category_clean"] = df["raw_category"].fillna("").str.lower().str.strip()

    print(f"\nAfter dedup: {len(df)} rows ready for attribution.")
    return df


# ── Step 2: Build Lookup Structures ─────────────────────────────────────────

def build_lookups(taxonomy: dict) -> tuple:
    """
    Preprocess taxonomy into fast lookup structures.
    Returns keyword_map and brand_map.
    """
    keyword_map = {}   # keyword → category
    brand_map   = {}   # brand   → category

    for category, rules in taxonomy.items():
        for kw in rules.get("keywords", []):
            keyword_map[kw.lower().strip()] = category
        for br in rules.get("brands", []):
            brand_map[br.lower().strip()] = category

    return keyword_map, brand_map


# ── Step 3: Score a Single Product ──────────────────────────────────────────

def score_product(name_clean: str,
                  brand_clean: str,
                  raw_cat_clean: str,
                  keyword_map: dict,
                  brand_map: dict) -> tuple:
    """
    Returns (mapped_category, confidence, match_reason).

    Logic:
      1. Score product name + raw_category text against all keywords
      2. Pick highest-scoring category → HIGH confidence
      3. If no keyword match, try brand lookup → MEDIUM confidence
      4. Else → NEEDS_REVIEW (LOW)

    ASSUMPTION: raw_category text is additional signal, not ground truth —
    clients often send messy or inconsistent category labels.
    """
    # Combine name + raw category as the text to search
    search_text = f"{name_clean} {raw_cat_clean}"

    # Score each category
    scores = {}
    matched_keywords = {}
    for kw, cat in keyword_map.items():
        if kw in search_text:
            scores[cat] = scores.get(cat, 0) + 1
            matched_keywords.setdefault(cat, []).append(kw)

    if scores and max(scores.values()) >= HIGH_THRESHOLD:
        best_cat = max(scores, key=scores.get)
        return (
            best_cat,
            "HIGH",
            f"keyword match: {matched_keywords[best_cat]}"
        )

    # Brand-level fallback
    if brand_clean in brand_map:
        return (
            brand_map[brand_clean],
            "MEDIUM",
            f"brand inference: '{brand_clean}'"
        )

    return ("NEEDS_REVIEW", "LOW", "no keyword or brand match")


# ── Step 4: Run Attribution Across All Products ──────────────────────────────

def run_attribution(df: pd.DataFrame,
                    taxonomy: dict) -> pd.DataFrame:
    """
    Apply scoring to every product and return enriched DataFrame.
    """
    print("\n" + "="*60)
    print("STEP 2: RUNNING ATTRIBUTION ENGINE")
    print("="*60)

    keyword_map, brand_map = build_lookups(taxonomy)

    results = df.apply(
        lambda row: score_product(
            row["product_name_clean"],
            row["brand_clean"],
            row["raw_category_clean"],
            keyword_map,
            brand_map
        ),
        axis=1,
        result_type="expand"
    )

    results.columns = ["mapped_category", "confidence", "match_reason"]
    df = pd.concat([df, results], axis=1)

    # Summary
    print(f"\nAttribution complete. Results:")
    conf_counts = df["confidence"].value_counts()
    for level in ["HIGH", "MEDIUM", "LOW"]:
        count = conf_counts.get(level, 0)
        pct = round(count / len(df) * 100, 1)
        print(f"  {level:<8}: {count:>4} products ({pct}%)")

    print(f"\nCategory distribution:")
    print(df["mapped_category"].value_counts().to_string())

    return df


# ── Step 5: Generate Output ──────────────────────────────────────────────────

def save_outputs(df: pd.DataFrame):
    """
    Write attributed CSV and a plain-text summary report.
    """
    print("\n" + "="*60)
    print("STEP 3: SAVING OUTPUTS")
    print("="*60)

    os.makedirs("output", exist_ok=True)

    # Select and order output columns
    output_cols = [
        "product_id", "product_name", "brand",
        "raw_category", "mapped_category",
        "confidence", "match_reason",
        "price_inr", "stock_units"
    ]
    out_df = df[[c for c in output_cols if c in df.columns]]
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"  Attributed data → {OUTPUT_CSV}")

    # Plain-text report
    total       = len(df)
    high_count  = (df["confidence"] == "HIGH").sum()
    med_count   = (df["confidence"] == "MEDIUM").sum()
    low_count   = (df["confidence"] == "LOW").sum()
    review_pct  = round(low_count / total * 100, 1)

    cat_dist = df["mapped_category"].value_counts()

    report_lines = [
        "PRODUCT ATTRIBUTION REPORT",
        f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "="*50,
        "",
        "SUMMARY",
        f"  Total products processed : {total}",
        f"  HIGH confidence          : {high_count} ({round(high_count/total*100,1)}%)",
        f"  MEDIUM confidence        : {med_count} ({round(med_count/total*100,1)}%)",
        f"  LOW / Needs Review       : {low_count} ({review_pct}%)",
        "",
        "CATEGORY DISTRIBUTION",
    ]
    for cat, count in cat_dist.items():
        report_lines.append(f"  {cat:<20} : {count}")

    report_lines += [
        "",
        "FLAGGED FOR MANUAL REVIEW",
    ]
    flagged = df[df["confidence"] == "LOW"][["product_id", "product_name", "brand"]]
    for _, row in flagged.iterrows():
        report_lines.append(f"  [{row['product_id']}] {row['product_name']} (brand: {row['brand']})")

    report_lines += [
        "",
        "ASSUMPTIONS",
        "  1. Exact duplicate rows removed before attribution.",
        "  2. Raw category field treated as supplementary signal, not ground truth.",
        "  3. Brand inference used only when keyword scoring fails.",
        "  4. Products with no keyword or brand match flagged LOW for analyst review.",
        "  5. Keyword threshold set to 1 match; configurable in taxonomy JSON.",
        "",
        "RECOMMENDATIONS",
        f"  - Review {low_count} LOW-confidence products manually.",
        "  - Consider expanding taxonomy keywords for out-of-scope brands.",
        "  - A TF-IDF semantic layer could reduce MEDIUM→HIGH reclassification effort.",
    ]

    with open(OUTPUT_REPORT, "w") as f:
        f.write("\n".join(report_lines))
    print(f"  Summary report     → {OUTPUT_REPORT}")


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
   

    # Load data
    df = load_and_audit(INPUT_PRODUCTS)

    # Load taxonomy
    with open(INPUT_TAXONOMY) as f:
        taxonomy = json.load(f)
    print(f"\nLoaded taxonomy with {len(taxonomy)} categories: {list(taxonomy.keys())}")

    # Run attribution
    df = run_attribution(df, taxonomy)

    # Save outputs
    save_outputs(df)

    print("\nDone. Check the output/ folder.")


if __name__ == "__main__":
    main()


STEP 1: LOADING & AUDITING DATA

Shape          : 46 rows × 6 columns
Columns        : ['product_id', 'product_name', 'brand', 'raw_category', 'price_inr', 'stock_units']

Null counts:
product_id       0
product_name     0
brand            3
raw_category    16
price_inr        0
stock_units      0

Duplicate rows : 1
Duplicate IDs  : 1

⚠  Dropped 1 exact duplicate row(s)

After dedup: 45 rows ready for attribution.

Loaded taxonomy with 4 categories: ['Skincare', 'Haircare', 'Bath & Body', 'Oral Care']

STEP 2: RUNNING ATTRIBUTION ENGINE

Attribution complete. Results:
  HIGH    :   38 products (84.4%)
  MEDIUM  :    2 products (4.4%)
  LOW     :    5 products (11.1%)

Category distribution:
mapped_category
Skincare        12
Haircare        10
Bath & Body     10
Oral Care        8
NEEDS_REVIEW     5

STEP 3: SAVING OUTPUTS
  Attributed data → output/attributed_products.csv
  Summary report     → output/attribution_report.txt

Done. Check the output/ folder.


In [4]:
import os
print(os.path.abspath("output"))

c:\Users\sadia\AppData\Local\Programs\Microsoft VS Code\output
